# Dados e Aprendizagem Automática 26/27
### Part VIII

## **Multi Layer Percepreton using Titanic Dataset - Model 2**

#### Imports

In [ ]:
import sklearn as skl
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch

from torch import Tensor
from torch.utils.data import Dataset
from torch.utils.data import DataLoader
from torch.utils.data import random_split
from torch.nn import Module, Linear, ReLU, Softmax, CrossEntropyLoss
from torch.nn.init import kaiming_uniform_, xavier_uniform_
from torchinfo import summary
from torch.optim import SGD, Adam

from IPython.display import display

from livelossplot import PlotLosses

from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

%matplotlib inline

device = torch.device("cpu")

#### Data Preparation

In [ ]:
class CSVDataset():
    def __init__(self):
        df_X = pd.read_csv("titanic_X.csv", header=0)
        df_y = pd.read_csv("titanic_y.csv", header=0)
        self.X = df_X.values
        self.y = df_y.values[:, 0]-1
        self.X = self.X.astype('float32')
        self.y = torch.tensor(self.y, dtype=torch.long, device=device)
        print(self.X.shape)
        print(self.y.shape)
        print(self.X.ndim)
        print(self.y.ndim)
        print(self.X.dtype)
        print(self.y.dtype)
        
    def __len__(self):
        return len(self.X)
 
    def __getitem__(self, idx):
        return [self.X[idx], self.y[idx]]
 
    def get_splits(self, n_test, n_val):
        test_size = round(n_test * len(self.X))
        val_size = round(n_val * len(self.X))
        train_size = len(self.X) - (test_size + val_size)
        return random_split(self, [train_size, test_size, val_size])
    
def prepare_data(n_test, n_val):
    dataset = CSVDataset()
    train, test, val = dataset.get_splits(n_test, n_val)
    train_dl = DataLoader(train, batch_size=len(train), shuffle=True)
    test_dl = DataLoader(test, batch_size=len(test), shuffle=True)
    val_dl = DataLoader(val, batch_size=len(val), shuffle=True)
    return train_dl, test_dl, val_dl

In [ ]:
train_dl, test_dl, val_dl = prepare_data(0.1, 0.2)

#### MLP Model 2

- feedforward network with fully interconnected layers
- **epochs**: 200
- **learning rate**: 0.001
- **layers**: 3 Linear
- **initialisations**: kaiming and xavier
- **activation fuctions**: ReLU and Softmax
- **loss**: CrossEntropyLoss
- **optimiser**: SGD

In [ ]:
EPOCHS = 200
LEARNING_RATE = 0.001

In [ ]:
class MLP_2(Module):
    def __init__(self, n_inputs):
        super(MLP_2, self).__init__()
        self.hidden1 = Linear(n_inputs, 24)
        self.hidden2 = Linear(24, 12)
        self.hidden3 = Linear(12, 3)
        
       
        self.act1 = ReLU()
        self.act2 = Softmax(dim=1)
 
    def forward(self, X):
        X = self.hidden1(X)
        X = self.act1(X)
        X = self.hidden2(X)
        X = self.act1(X)
        X = self.hidden3(X)
        X = self.act2(X)
        return X

Define the neural network

In [ ]:
model = MLP_2()

Visualize the network

In [ ]:
print(summary(model, input_size=(len(train_dl.dataset), 8), verbose=0))
model.to(device)

#### Train the model

In [ ]:
def train_model(train_dl, val_dl, model):
    liveloss = PlotLosses()
    criterion = CrossEntropyLoss()
    optimizer = SGD(model.parameters(), lr=LEARNING_RATE, momentum=0.9)

    for epoch in range(EPOCHS):
        logs = {} 
        model.train()
        running_loss  = 0.0
        running_corrects  = 0.0
        for inputs, labels in train_dl: 
            inputs = inputs.to(device)
            labels = labels.to(device)
            
            outputs = model(inputs)
            
            loss = criterion(outputs, labels)
            optimizer.zero_grad() 
            loss.backward()
            optimizer.step()
            running_loss += loss.detach() * inputs.size(0)
            
            _, preds = torch.max(outputs, 1)
            running_corrects += torch.sum(preds == labels.data)
        
        epoch_loss = running_loss / len(train_dl.dataset)
        epoch_acc = running_corrects.float() / len(train_dl.dataset)
        logs['loss'] = epoch_loss.item()
        logs['accuracy'] = epoch_acc.item()

        model.eval()
        running_loss_val  = 0.0
        running_corrects_val  = 0.0
        for inputs, labels in val_dl: 
            inputs = inputs.to(device)
            labels = labels.to(device)
            
            outputs = model(inputs)
            
            loss_val = criterion(outputs, labels)
            running_loss_val += loss_val.detach() * inputs.size(0)
            _, preds = torch.max(outputs, 1))
            running_corrects_val += torch.sum(preds == labels.data)
        
        epoch_loss_val = running_loss_val / len(val_dl.dataset)
        epoch_acc_val = running_corrects_val.float() / len(val_dl.dataset)
        logs['val_loss'] = epoch_loss_val.item()
        logs['val_accuracy'] = epoch_acc_val.item()    
        liveloss.update(logs)
        liveloss.send()

In [ ]:
train_model(train_dl, val_dl, model)

#### Evaluate the model

In [ ]:
def evaluate_model(test_dl, model):
    predictions = list()
    actual_values = list()
    for i, (inputs, labels) in enumerate(test_dl):
        yprev = model(inputs)
        yprev = yprev.detach().numpy()
        actual = labels.numpy()
        yprev = np.argmax(yprev, axis=1)
        actual = actual.reshape((len(actual), 1))
        yprev = yprev.reshape((len(yprev), 1))
        predictions.append(yprev)
        actual_values.append(actual)
        break
    predictions, actual_values = np.vstack(predictions), np.vstack(actual_values)
    return predictions, actual_values

In [ ]:
def display_confusion_matrix(cm):
    plt.figure(figsize = (16,8))
    sns.heatmap(cm, annot=True, xticklabels=['Class 1', 'Class 2', 'Class 3'],
                yticklabels=['Class 1', 'Class 2', 'Class 3'], 
                annot_kws={"size": 12}, fmt='g', linewidths=.5)
    plt.ylabel('True label')
    plt.xlabel('Predicted label')
    plt.show() 

In [ ]:
predictions, actual_values = evaluate_model(test_dl, model)

In [ ]:
success = 0
failure = 0
for r,p in zip(actual_values, predictions):
    print(f'real:{r+1} prediction:{p+1}') 
    if r==p: success+=1  
    else: failure+=1

In [ ]:
acc = accuracy_score(actual_values, predictions)
print(f'Accuracy: {acc:0.3f}\n')
print(f'success:{success} failure:{failure}')

In [ ]:
print(classification_report(actual_values, predictions
cm = confusion_matrix(actual_values, predictions)
print(cm)
display_confusion_matrix(cm)

#### Apply the model

Make a prediction using one case:

In [ ]:
def predict(row, model):
    row = Tensor([row])
    yprev = model(row)
    yprev = yprev.detach().numpy()
    return yprev

In [ ]:
row = [5, 0, 1, 34, 0, 1, 8, 1]
yprev = predict(row, model)
print('Predicted: %s (class=%d)' % (yprev, np.argmax(yprev)+1))